In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading the data
#### Source: _Yahoo Finance_

In [ ]:
from data import download_tickers_history

# set the date range for the historic data
start_date = datetime(year=2020, month=1, day=1)
end_date = datetime(year=2025, month=12, day=31)
history = download_tickers_history(start_date, end_date, ['NVDA']);

nvda = history.NVDA;


## Intraday Features
Global models often do not support what happens within a single trading day.

So we can visualize and analyze the gaps, and fluctuations that are happening within a day.

In [ ]:
from data import overnight_gaps_prc

overnight_gap = overnight_gaps_prc(history).NVDA
dates = overnight_gap.index

### High-to-Low Spread
$$\text{TR}_t = \max\left( \text{High}_t - \text{Low}_t,\; \vert{}\text{High}_t - \text{Close}_{t-1}\vert{},\; \vert{}\text{Low}_t - \text{Close}_{t-1}\vert{} \right)$$
$$Spread_t = [(High_t - Low_t) / Close_t] * 100\%$$

In [ ]:
from data import daily_spread_pct, rolling_daily_spreads_mean

spread_pct = daily_spread_pct(history).NVDA
atr_14 = rolling_daily_spreads_mean(history, 14).NVDA
dates = spread_pct.index

# data visualization
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(
    dates,
    spread_pct,
    color='lightgreen',
    alpha=1,
    linewidth=1,
    label='Daily spread (%)'
);

plt.plot(
    dates,
    atr_14,
    color='darkred',
    alpha=0.7,
    linewidth=2,
    label='14-Day Average Range (ATR Trend)'
);

plt.title('NVDA: Daily price change (%)')
plt.xlabel('Periods (1 period = 1 day)', fontsize=12)
plt.ylabel('Daily Change (% of close prices)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()


### Open-to-Close (Overnight) Gap
Shows how much risk the market carries overnight while the exchange is closed.
$$ln(Open_t / Close_{t-1})$$

In [ ]:
# data visualization
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(
    dates,
    overnight_gap,
    color='darkblue',
    alpha=0.7,
    linewidth=2.5,
    label='Nightly price shift (%)'
);

plt.title('NVDA: Overnight gap (%)')
plt.xlabel('Periods (1 period = 1 day)', fontsize=12)
plt.ylabel('Overnight Change (% from Previous Close)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()

If _Skewness_ $> 0$, positive gaps (upwards) occur more frequently or are more aggressive.

A _Kurtosis_ value $> 3$ will immediately show heavy-tail risk, the probability of waking up with a $-10\%$ gap.

In [ ]:
gap_skew = overnight_gap.skew()
gap_kurt = overnight_gap.kurtosis()

fig, ax = plt.subplots(figsize=(12, 6))

plt.hist(
    overnight_gap, 
    density=True,
    bins=200,
    linewidth=1,
    color='green',
    edgecolor='w',
    label='Open-Close Gap'
)

plt.title(f'NVDA: Overnight gap distr (Skew: {gap_skew:.2f}, Kurt: {gap_kurt:.2f})')
plt.xlabel('Overnight gap (%)', fontsize=12)
plt.ylabel('Distribution Density', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)

plt.show()

#### Overnight Volatility (std)

In [ ]:
from data import rolling_overnight_gaps_std

overnight_vol_20 = rolling_overnight_gaps_std(history, 20).NVDA # 20-day window of overnight gap volatility
dates = overnight_vol_20.index

fig, ax = plt.subplots(figsize=(12, 6))

plt.plot(
    dates,
    overnight_gap,
    linewidth=1,
    color='purple',
    label='20-Day rolling overnight volatility (std)'
)

plt.title(f'NVDA: Overnight Volatility')
plt.xlabel('Period', fontsize=12)
plt.ylabel('Std Dev (%)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)

plt.show()


#### Gap Continuation / Mean Reversion
The main metric for *swing trading* is what happens to the price *AFTER* a gap during the trading day:

* **Fade**: If the price opened with an upward gap, but then fell back to the previous day's close.
* **Continuation**: If after an upward gap, the price continues to rise until the close.

##### Strategies
* If the correlation between gap and the last day's return is **negative** &mdash; the major strategy on the market is the _"filling the gaps"_ (sell the gap up)
* If the correlation between gap and the last day's return is **positive** &mdash; the major _"impulsive strategy"_ is the dominant on the market (buy the gap up, since the price continues to grow)

In [ ]:
import seaborn as sns
from colorama import init, Style
from data import intraday_returns_prc

init(autoreset=True)

intraday_return = intraday_returns_prc(history)[:-1].NVDA # trim last day to get equal length with overnight_gap
dates = intraday_return.index.year

gap_data = pd.DataFrame({
    'Overnight_Gap': overnight_gap,
    'Intraday_Return': intraday_return,
}).dropna()

# Pearson correlation
p_corr = gap_data['Overnight_Gap'].corr(gap_data['Intraday_Return'])
print(f"Pearson correlation (Linear): {Style.BRIGHT} {p_corr:.3f}")

# Trim the outliers
p_low = gap_data['Overnight_Gap'].quantile(0.01)
p_high = gap_data['Overnight_Gap'].quantile(0.99)

filtered_data = gap_data[
    (gap_data['Overnight_Gap'] > p_low) & 
    (gap_data['Overnight_Gap'] < p_high)
]

print(f"Pearson correlation (no outliers): {Style.BRIGHT} {filtered_data['Overnight_Gap'].corr(filtered_data['Intraday_Return']):.3f}")

# Spearman correlation
p_corr = gap_data['Overnight_Gap'].corr(gap_data['Intraday_Return'], method='spearman')
print(f"Spearman correlation (Rank): {Style.BRIGHT} {p_corr:.3f}")

# data visualization
fig, ax = plt.subplots(figsize=(11, 7))

scatter = plt.scatter(
    overnight_gap,
    intraday_return,
    c=dates,
    cmap="viridis",
    s=35
)

# zero lines for Quadrants
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)

plt.title(f'NVDA: Overnight Gap vs. Intraday Return (Fade / Continuation)')

# linear trendline
sns.regplot(
    x='Overnight_Gap',
    y='Intraday_Return',
    data=gap_data,
    ax=ax,
    scatter=False,
    color='red',
    line_kws={'linewidth': 1.8, 'label': f'OLS Trend (r = {p_corr:.3f})'}
)

legend1 = ax.legend(*scatter.legend_elements(num=6),
                    loc="lower right", title="Gap shift (%)")
ax.add_artist(legend1)

# colorbar for Years
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Year', fontsize=11)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()


## Overnight vs. Intraday
$$\text{Ratio} = \frac{\text{Var}(\text{Overnight Returns})}{\text{Var}(\text{Intraday Returns})}$$

In a majority of a public corporations a portion of the total annual return comes from the overnight gaps (in earnings and announcements), not from intraday trading.
If the variance ratio is high, holding an overnight yields more risk/return than day trading.

In [ ]:
ratio = gap_data['Overnight_Gap'].var() / gap_data['Intraday_Return'].var()
print(f"Overnight vs Intraday Ratio: {ratio:.3f}")


### Close Location
Shows where the day closed relative to its low and high:
$$\frac{(Close_t - Low_t)}{(High_t - Low_t)}$$
A value close to 1 means that bulls (buyers) dominated at the end of the day.

In [ ]:
num_of_years = len(nvda.index) // 365
last_year_data = nvda[(num_of_years - 1) * 365:]

close_loc = (last_year_data['Close'] - last_year_data['Low']) / (last_year_data['High'] - last_year_data['Low'])
dates = last_year_data.index

# data visualization
fig, ax = plt.subplots(figsize=(12, 6))

ax.stem(
    dates,
    close_loc,
    linefmt='y-',
    markerfmt='go',
    label='Daily close localization'
);

plt.title('NVDA: Overnight Location')
plt.xlabel('Periods', fontsize=12)
plt.ylabel('Abs Overnight Location', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()
